<a href="https://colab.research.google.com/github/rayana11-blip/Technology_Usage_Analytics/blob/main/GitProject_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# 1. IMPORT LIBRARIES

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
import re

from scipy import stats
from google.colab import files
from IPython.display import display

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

# Visualization settings
sns.set_theme(style='whitegrid')

print("All libraries have been imported successfully!")

All libraries have been imported successfully!


In [2]:
# 2. UPLOAD DATASET

uploaded = files.upload()

filename = next(iter(uploaded))

df_raw = pd.read_csv(filename)

print("Dataset uploaded successfully!")
print("File name:", filename)
print("Number of rows:", df_raw.shape[0])
print("Number of columns:", df_raw.shape[1])

display(df_raw.head())

Saving Technology_Usage_Survey_Responses_English.csv to Technology_Usage_Survey_Responses_English.csv
Dataset uploaded successfully!
File name: Technology_Usage_Survey_Responses_English.csv
Number of rows: 55
Number of columns: 6


,ID,Frequently Used Devices,Purpose of Use,User Age,Time Spent per Day (hours),Monthly Data Usage (GB)
0,ID001,Smartphone,Entertainment,4 years,8 hours,5 GB
1,ID002,Smartphone,Communication tool & learning media,18,7,50 GB
2,ID003,"Smartphone, Laptop, Tablet",Entertainment,18 years,6 hours,10 GB
3,ID004,Smartphone,Communication,18,9 hours,20 GB
4,ID005,Smartphone,Communication,18,6,Uses Wi-Fi


In [3]:
# 3. CHECK DATASET STRUCTURE

print("DATASET DIMENSIONS")
print("Number of respondents:", df_raw.shape[0])
print("Number of variables:", df_raw.shape[1])

print("\nCOLUMN NAMES")

for index, column in enumerate(df_raw.columns, start=1):
    print(index, repr(column))

print("\nDATA TYPES")

display(df_raw.dtypes.to_frame("Data Type"))

print("\nDATASET INFORMATION")

df_raw.info()


DATASET DIMENSIONS
Number of respondents: 55
Number of variables: 6

COLUMN NAMES
1 'ID'
2 'Frequently Used Devices'
3 'Purpose of Use'
4 'User Age'
5 'Time Spent per Day (hours)'
6 'Monthly Data Usage (GB)'

DATA TYPES


,Data Type
ID,object
Frequently Used Devices,object
Purpose of Use,object
User Age,object
Time Spent per Day (hours),object
Monthly Data Usage (GB),object



DATASET INFORMATION
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 55 entries, 0 to 54
Data columns (total 6 columns):
 #   Column                      Non-Null Count  Dtype 
---  ------                      --------------  ----- 
 0   ID                          55 non-null     object
 1   Frequently Used Devices     55 non-null     object
 2   Purpose of Use              55 non-null     object
 3   User Age                    54 non-null     object
 4   Time Spent per Day (hours)  55 non-null     object
 5   Monthly Data Usage (GB)     55 non-null     object
dtypes: object(6)
memory usage: 2.7+ KB


In [4]:
# 4. CREATE DATAFRAME AND RENAME COLUMNS

df = df_raw.copy()

rename_map = {
    'ID': 'id',
    'Frequently Used Devices': 'devices',
    'Purpose of Use': 'purpose',
    'User Age': 'user_age',
    'Time Spent per Day (hours)': 'daily_hours',
    'Monthly Data Usage (GB)': 'monthly_data_gb'
}

df = df.rename(columns=rename_map)

print("Column names updated successfully!")

display(df.head())

print("\nUpdated column names:")
print(df.columns.tolist())


Column names updated successfully!


,id,devices,purpose,user_age,daily_hours,monthly_data_gb
0,ID001,Smartphone,Entertainment,4 years,8 hours,5 GB
1,ID002,Smartphone,Communication tool & learning media,18,7,50 GB
2,ID003,"Smartphone, Laptop, Tablet",Entertainment,18 years,6 hours,10 GB
3,ID004,Smartphone,Communication,18,9 hours,20 GB
4,ID005,Smartphone,Communication,18,6,Uses Wi-Fi



Updated column names:
['id', 'devices', 'purpose', 'user_age', 'daily_hours', 'monthly_data_gb']


In [5]:
# 5. CHECK MISSING VALUES

missing_summary = pd.DataFrame({
    'Missing Values': df.isnull().sum(),
    'Missing Percentage (%)': (
        df.isnull().mean() * 100
    ).round(2)
})

display(missing_summary)

print("Total missing values:", df.isnull().sum().sum())


,Missing Values,Missing Percentage (%)
id,0,0.00
devices,0,0.00
purpose,0,0.00
user_age,1,1.82
daily_hours,0,0.00
monthly_data_gb,0,0.00


Total missing values: 1


In [6]:
# 6. CHECK DUPLICATED DATAS

print("Duplicate rows:", df.duplicated().sum())

print("Duplicate IDs:", df['id'].duplicated().sum())

duplicate_data = df[
    df.duplicated(keep=False)
]

print("\nDuplicate records:")

display(duplicate_data)

Duplicate rows: 0
Duplicate IDs: 0

Duplicate records:


,id,devices,purpose,user_age,daily_hours,monthly_data_gb


In [7]:
# 7. CHECK UNIQUE VALUES

for column in df.columns:

    print("\n================================")
    print("VARIABLE:", column)
    print("================================")

    display(
        df[column].value_counts(dropna=False)
    )


VARIABLE: id


,count
id,
ID001,1
ID002,1
ID003,1
ID004,1
ID005,1
ID006,1
ID007,1
ID008,1
ID009,1



VARIABLE: devices


,count
devices,
Smartphone,34
"Smartphone, Laptop",11
Tablet,5
"Smartphone, Tablet",2
"Smartphone, Laptop, Tablet",1
Laptop,1
"Laptop, Tablet",1



VARIABLE: purpose


,count
purpose,
Communication,27
Entertainment,14
Learning media,12
Communication tool & learning media,1
All of the above,1


... (2 outputs left out)


VARIABLE: daily_hours


,count
daily_hours,
6,5
6 hours,4
7,4
15,4
9,3
20,3
10,3
10 hours,3
8 hours,2



VARIABLE: monthly_data_gb


,count
monthly_data_gb,
30,3
5 GB,2
20 GB,2
50 GB,2
50,2
35,2
20,2
Uses Wi-Fi,2
15 GB,2


In [8]:
# 8. CHECK ORIGINAL RESPONSES

print("USER AGE")
display(df['user_age'].value_counts(dropna=False))

print("\nDAILY USAGE DURATION")
display(df['daily_hours'].value_counts(dropna=False))

print("\nMONTHLY DATA USAGE")
display(df['monthly_data_gb'].value_counts(dropna=False))

USER AGE


,count
user_age,
18,15
19,10
17,8
18 years,3
20,3
19 years,2
16,2
20 years,2
4 years,1



DAILY USAGE DURATION


,count
daily_hours,
6,5
6 hours,4
7,4
15,4
9,3
20,3
10,3
10 hours,3
8 hours,2



MONTHLY DATA USAGE


,count
monthly_data_gb,
30,3
5 GB,2
20 GB,2
50 GB,2
50,2
35,2
20,2
Uses Wi-Fi,2
15 GB,2


In [9]:
# 9. CLEAN TEXT DATA

text_columns = [
    'devices',
    'purpose',
    'user_age',
    'daily_hours',
    'monthly_data_gb'
]

for column in text_columns:

    df[column] = (
        df[column]
        .astype('string')
        .str.strip()
    )

# Convert empty strings into missing values
df = df.replace(r'^\s*$', pd.NA, regex=True)

print("Text cleaning completed!")

display(df.head())

Text cleaning completed!


,id,devices,purpose,user_age,daily_hours,monthly_data_gb
0,ID001,Smartphone,Entertainment,4 years,8 hours,5 GB
1,ID002,Smartphone,Communication tool & learning media,18,7,50 GB
2,ID003,"Smartphone, Laptop, Tablet",Entertainment,18 years,6 hours,10 GB
3,ID004,Smartphone,Communication,18,9 hours,20 GB
4,ID005,Smartphone,Communication,18,6,Uses Wi-Fi


In [10]:
# 10. CLEAN USER AGE

def clean_age(value):

    if pd.isna(value):
        return np.nan

    text = str(value).lower().strip()

    match = re.search(r'\d+', text)

    if match:

        age = int(match.group())

        # Treat the questionable age of 4 as missing
        if age == 4:
            return np.nan

        return age

    return np.nan


df['age_years'] = df['user_age'].apply(clean_age)

print("Age cleaning completed!")

display(
    df[['id', 'user_age', 'age_years']]
)

Age cleaning completed!


,id,user_age,age_years
0,ID001,4 years,NaN
1,ID002,18,18.0
2,ID003,18 years,18.0
3,ID004,18,18.0
4,ID005,18,18.0
5,ID006,18,18.0
6,ID007,17,17.0
7,ID008,18 years,18.0
8,ID009,18,18.0
9,ID010,19,19.0


In [11]:
# 11. CHECK AGE VALUES

print("Missing age values:")

display(
    df[
        df['age_years'].isna()
    ][['id', 'user_age', 'age_years']]
)

print("\nValid age records:", df['age_years'].count())

print("Total respondents:", len(df))

Missing age values:


,id,user_age,age_years
0,ID001,4 years,NaN
52,ID053,<NA>,NaN



Valid age records: 53
Total respondents: 55


In [12]:
# 12. CLEAN DAILY USAGE DURATION

def clean_duration(value):

    if pd.isna(value):
        return np.nan

    text = str(value).lower().strip()

    # Ambiguous upper-bound response
    if '≤' in text or 'less than' in text:
        return np.nan

    text = text.replace('hours', '')
    text = text.replace('hour', '')
    text = text.replace('maybe', '')
    text = text.strip()

    # Detect numerical ranges
    match = re.search(
        r'(\d+(?:[.,]\d+)?)\s*-\s*(\d+(?:[.,]\d+)?)',
        text
    )

    if match:

        number_1 = float(match.group(1).replace(',', '.'))
        number_2 = float(match.group(2).replace(',', '.'))

        return (number_1 + number_2) / 2

    # Detect single numerical values
    match = re.search(r'\d+(?:[.,]\d+)?', text)

    if match:
        return float(match.group().replace(',', '.'))

    return np.nan


df['daily_hours_clean'] = (
    df['daily_hours'].apply(clean_duration)
)

print("Daily duration cleaning completed!")

display(
    df[['id', 'daily_hours', 'daily_hours_clean']]
)

Daily duration cleaning completed!


,id,daily_hours,daily_hours_clean
0,ID001,8 hours,8.0
1,ID002,7,7.0
2,ID003,6 hours,6.0
3,ID004,9 hours,9.0
4,ID005,6,6.0
5,ID006,8,8.0
6,ID007,8 hours,8.0
7,ID008,1-2 hours,1.5
8,ID009,3-5 hours,4.0
9,ID010,9,9.0


In [13]:
# 13. CLEAN MONTHLY DATA USAGE

def clean_data_usage(value):

    if pd.isna(value):
        return np.nan

    text = str(value).lower().strip()

    # Wi-Fi responses do not provide numerical GB usage
    if 'wifi' in text or 'wi-fi' in text:
        return np.nan

    # Reject negative values
    if re.match(r'^-\s*\d+', text):
        return np.nan

    # Handle numerical ranges
    match = re.search(
        r'(\d+(?:[.,]\d+)?)\s*-\s*(\d+(?:[.,]\d+)?)',
        text
    )

    if match:

        number_1 = float(match.group(1).replace(',', '.'))
        number_2 = float(match.group(2).replace(',', '.'))

        return (number_1 + number_2) / 2

    # Reject ambiguous date-like responses
    if re.match(r'^\d{1,2}-[a-z]{3}$', text):
        return np.nan

    # Handle single numerical values
    match = re.search(r'\d+(?:[.,]\d+)?', text)

    if match:
        return float(match.group().replace(',', '.'))

    return np.nan


df['monthly_data_gb_clean'] = (
    df['monthly_data_gb'].apply(clean_data_usage)
)

print("Monthly data usage cleaning completed!")

display(
    df[
        ['id', 'monthly_data_gb', 'monthly_data_gb_clean']
    ]
)

Monthly data usage cleaning completed!


,id,monthly_data_gb,monthly_data_gb_clean
0,ID001,5 GB,5.0
1,ID002,50 GB,50.0
2,ID003,10 GB,10.0
3,ID004,20 GB,20.0
4,ID005,Uses Wi-Fi,NaN
5,ID006,45,45.0
6,ID007,50,50.0
7,ID008,3-5 GB,4.0
8,ID009,50 GB,50.0
9,ID010,25,25.0


In [14]:
# 14. CHECK CLEANED DATA

cleaned_columns = [
    'age_years',
    'daily_hours_clean',
    'monthly_data_gb_clean'
]

cleaning_summary = pd.DataFrame({
    'Valid Records': df[cleaned_columns].count(),
    'Missing Values': df[cleaned_columns].isna().sum(),
    'Total Respondents': len(df)
})

display(cleaning_summary)


,Valid Records,Missing Values,Total Respondents
age_years,53,2,55
daily_hours_clean,54,1,55
monthly_data_gb_clean,51,4,55


In [15]:
# 15. AUDIT UNCONVERTED RESPONSES

print("UNCONVERTED AGE RESPONSES")

display(
    df[
        df['age_years'].isna()
    ][['id', 'user_age']]
)

print("\nUNCONVERTED DURATION RESPONSES")

display(
    df[
        df['daily_hours_clean'].isna()
    ][['id', 'daily_hours']]
)

print("\nUNCONVERTED DATA USAGE RESPONSES")

display(
    df[
        df['monthly_data_gb_clean'].isna()
    ][['id', 'monthly_data_gb']]
)

UNCONVERTED AGE RESPONSES


,id,user_age
0,ID001,4 years
52,ID053,<NA>



UNCONVERTED DURATION RESPONSES


,id,daily_hours
16,ID017,≤10



UNCONVERTED DATA USAGE RESPONSES


,id,monthly_data_gb
4,ID005,Uses Wi-Fi
13,ID014,Uses Wi-Fi
31,ID032,6-Apr
41,ID042,-45


In [16]:
# 16. DESCRIPTIVE STATISTICS

numeric_columns = [
    'age_years',
    'daily_hours_clean',
    'monthly_data_gb_clean'
]

descriptive_stats = df[numeric_columns].describe().T

descriptive_stats['mode'] = [
    df[column].mode().iloc[0]
    if not df[column].mode().empty
    else np.nan
    for column in numeric_columns
]

display(descriptive_stats.round(2))

,count,mean,std,min,25%,50%,75%,max,mode
age_years,53.0,18.96,4.26,13.0,18.0,18.0,19.00,46.0,18.0
daily_hours_clean,54.0,9.85,5.01,1.5,6.0,9.0,13.75,24.0,6.0
monthly_data_gb_clean,51.0,37.29,31.00,2.0,16.0,30.0,50.00,150.0,30.0


In [17]:
# 17. PREPARE EDA DATA

print("Available columns:")

for column in df.columns:
    print(column)

print("\nEDA dataset is ready!")

print("Total respondents:", len(df))

Available columns:
id
devices
purpose
user_age
daily_hours
monthly_data_gb
age_years
daily_hours_clean
monthly_data_gb_clean

EDA dataset is ready!
Total respondents: 55


In [18]:
# 18. DEVICE USAGE DISTRIBUTIONS

device_counts = (
    df['devices']
    .value_counts()
    .reset_index()
)

device_counts.columns = ['Device', 'Count']

fig = px.bar(
    device_counts,
    x='Device',
    y='Count',
    text='Count',
    title='Frequently Used Devices',
    color_discrete_sequence=['#00BFFF']
)

fig.update_traces(
    marker_line_color='black',
    marker_line_width=1
)

fig.update_layout(
    xaxis_title='Device Type',
    yaxis_title='Number of Respondents',
    template='plotly_white'
)

fig.show()

In [19]:
# 19. PURPOSE OF TECHNOLOGY USAGE

purpose_counts = (
    df['purpose']
    .value_counts()
    .reset_index()
)

purpose_counts.columns = ['Purpose', 'Count']

fig = px.bar(
    purpose_counts,
    x='Purpose',
    y='Count',
    text='Count',
    title='Purpose of Technology Usage',
    color_discrete_sequence=['#00BFFF']
)

fig.update_traces(
    marker_line_color='black',
    marker_line_width=1
)

fig.update_layout(
    xaxis_title='Purpose',
    yaxis_title='Number of Respondents',
    template='plotly_white'
)

fig.show()

In [20]:
# 20. AGE DISTRIBUTIONS

age_data = df.dropna(subset=['age_years'])

fig = px.histogram(
    age_data,
    x='age_years',
    nbins=15,
    title='Age Distribution of Respondents',
    labels={
        'age_years': 'Age (Years)',
        'count': 'Number of Respondents'
    },
    color_discrete_sequence=['#00FFFF']
)

# Add black borders to histogram bars
fig.update_traces(
    marker_line_color='black',
    marker_line_width=1
)

fig.update_layout(
    xaxis_title='Age (Years)',
    yaxis_title='Number of Respondents',
    template='plotly_white'
)

fig.show()

In [22]:
# 21. DAILY USAGE DISTRIBUTION

duration_data = df.dropna(subset=['daily_hours_clean'])

fig = px.histogram(
    duration_data,
    x='daily_hours_clean',
    nbins=12,
    title='Daily Technology Usage Duration',
    labels={
        'daily_hours_clean': 'Daily Usage Duration (Hours)',
        'count': 'Number of Respondents'
    },
    color_discrete_sequence=['#00BFFF']
)

# Add black borders
fig.update_traces(
    marker_line_color='black',
    marker_line_width=1
)

fig.update_layout(
    xaxis_title='Daily Usage Duration (Hours)',
    yaxis_title='Number of Respondents',
    template='plotly_white'
)

fig.show()

In [23]:
# 22. MONTHLY DATA USAGE DISTRIBUTION

data_usage = df.dropna(
    subset=['monthly_data_gb_clean']
)

fig = px.histogram(
    data_usage,
    x='monthly_data_gb_clean',
    nbins=12,
    title='Monthly Internet Data Usage',
    labels={
        'monthly_data_gb_clean': 'Monthly Data Usage (GB)',
        'count': 'Number of Respondents'
    },
    color_discrete_sequence=['#00FFFF']
)

# Add black borders
fig.update_traces(
    marker_line_color='black',
    marker_line_width=1
)

fig.update_layout(
    xaxis_title='Monthly Data Usage (GB)',
    yaxis_title='Number of Respondents',
    template='plotly_white'
)

fig.show()

In [24]:
# 23. OUTLIER DETECTION USING IQR

numeric_variables = {
    'Age': 'age_years',
    'Daily Usage Duration': 'daily_hours_clean',
    'Monthly Data Usage': 'monthly_data_gb_clean'
}

outlier_results = []

for variable_name, column in numeric_variables.items():

    values = df[column].dropna()

    Q1 = values.quantile(0.25)
    Q3 = values.quantile(0.75)

    IQR = Q3 - Q1

    lower_fence = Q1 - 1.5 * IQR
    upper_fence = Q3 + 1.5 * IQR

    outliers = df[
        (df[column] < lower_fence) |
        (df[column] > upper_fence)
    ]

    outlier_results.append({
        'Variable': variable_name,
        'Q1': Q1,
        'Q3': Q3,
        'IQR': IQR,
        'Lower Fence': lower_fence,
        'Upper Fence': upper_fence,
        'Outlier Count': len(outliers)
    })

    print(f"\nOUTLIERS: {variable_name}")

    display(
        outliers[['id', column]]
    )

outlier_summary = pd.DataFrame(outlier_results)

print("\nOUTLIER SUMMARY")

display(outlier_summary.round(2))


OUTLIERS: Age


,id,age_years
12,ID013,16.0
16,ID017,23.0
17,ID018,16.0
22,ID023,15.0
24,ID025,46.0
26,ID027,25.0
28,ID029,22.0
42,ID043,24.0
49,ID050,13.0



OUTLIERS: Daily Usage Duration


,id,daily_hours_clean



OUTLIERS: Monthly Data Usage


,id,monthly_data_gb_clean
30,ID031,120.0
46,ID047,150.0



OUTLIER SUMMARY


,Variable,Q1,Q3,IQR,Lower Fence,Upper Fence,Outlier Count
0,Age,18.0,19.00,1.00,16.50,20.50,9
1,Daily Usage Duration,6.0,13.75,7.75,-5.62,25.38,0
2,Monthly Data Usage,16.0,50.00,34.00,-35.00,101.00,2


In [26]:
# 24. BOXPLOT ANALYSIS

fig = go.Figure()

boxplot_variables = [
    ('Age', 'age_years'),
    ('Daily Usage (Hours)', 'daily_hours_clean'),
    ('Monthly Data Usage (GB)', 'monthly_data_gb_clean')
]

for label, column in boxplot_variables:

    fig.add_trace(
        go.Box(
            y=df[column].dropna(),
            name=label,
            boxpoints='outliers'
        )
    )

fig.update_layout(
    title='Boxplot Analysis of Technology Usage',
    yaxis_title='Observed Values',
    template='plotly_white',
    height=500
)

fig.show()

In [27]:
# 25. DAILY USAGE VS MONTHLY DATA USAGE

scatter_data = df.dropna(
    subset=[
        'daily_hours_clean',
        'monthly_data_gb_clean'
    ]
).copy()

fig = px.scatter(
    scatter_data,
    x='daily_hours_clean',
    y='monthly_data_gb_clean',
    color='purpose',
    hover_data=['id', 'age_years'],
    title='Daily Usage Duration vs Monthly Data Usage',
    labels={
        'daily_hours_clean': 'Daily Usage Duration (Hours)',
        'monthly_data_gb_clean': 'Monthly Data Usage (GB)',
        'purpose': 'Purpose of Use'
    }
)

fig.update_traces(
    marker=dict(
        size=11,
        line=dict(
            color='black',
            width=1
        )
    )
)

fig.update_layout(
    template='plotly_white',
    height=550
)

fig.show()

print("Number of valid observations:", len(scatter_data))

Number of valid observations: 50


In [28]:
# 26. CORRELATION ANALYSIS

correlation_pairs = [
    ('age_years', 'daily_hours_clean'),
    ('age_years', 'monthly_data_gb_clean'),
    ('daily_hours_clean', 'monthly_data_gb_clean')
]

correlation_results = []

for variable_1, variable_2 in correlation_pairs:

    pair_data = df[
        [variable_1, variable_2]
    ].dropna()

    if len(pair_data) < 3:
        print(
            f"Not enough observations for "
            f"{variable_1} and {variable_2}"
        )
        continue

    pearson_r, pearson_p = stats.pearsonr(
        pair_data[variable_1],
        pair_data[variable_2]
    )

    spearman_r, spearman_p = stats.spearmanr(
        pair_data[variable_1],
        pair_data[variable_2]
    )

    correlation_results.append({
        'Variable 1': variable_1,
        'Variable 2': variable_2,
        'Valid N': len(pair_data),
        'Pearson r': pearson_r,
        'Pearson p-value': pearson_p,
        'Spearman rho': spearman_r,
        'Spearman p-value': spearman_p
    })

correlation_table = pd.DataFrame(correlation_results)

print("CORRELATION ANALYSIS RESULTS")

display(correlation_table.round(4))

CORRELATION ANALYSIS RESULTS


,Variable 1,Variable 2,Valid N,Pearson r,Pearson p-value,Spearman rho,Spearman p-value
0,age_years,daily_hours_clean,52,-0.0426,0.7645,-0.0578,0.6839
1,age_years,monthly_data_gb_clean,49,0.0608,0.6783,0.1976,0.1734
2,daily_hours_clean,monthly_data_gb_clean,50,0.2154,0.1330,0.2722,0.0558


In [29]:
# 27. CORRELATION HEATMAP

correlation_columns = [
    'age_years',
    'daily_hours_clean',
    'monthly_data_gb_clean'
]

correlation_matrix = df[
    correlation_columns
].corr(method='pearson')

fig = px.imshow(
    correlation_matrix,
    text_auto='.2f',
    color_continuous_scale='RdBu_r',
    zmin=-1,
    zmax=1,
    title='Correlation Heatmap of Technology Usage'
)

fig.update_layout(
    template='plotly_white',
    height=500
)

fig.show()

In [30]:
# 28. CALCULATE KEY PERFORMANCE INDICATORS

total_respondents = len(df)

average_age = df['age_years'].mean()

average_daily_usage = df['daily_hours_clean'].mean()

average_monthly_data = df['monthly_data_gb_clean'].mean()

print("==========================================")
print("     TECHNOLOGY USAGE ANALYTICS")
print("==========================================")

print("Total Respondents:",
      total_respondents)

print("Average Age:",
      round(average_age, 2), "years")

print("Average Daily Usage:",
      round(average_daily_usage, 2), "hours")

print("Average Monthly Data Usage:",
      round(average_monthly_data, 2), "GB")

print("==========================================")

     TECHNOLOGY USAGE ANALYTICS
Total Respondents: 55
Average Age: 18.96 years
Average Daily Usage: 9.85 hours
Average Monthly Data Usage: 37.29 GB


In [31]:
# 29. PREPARE DASHBOARD DATASET

dashboard_df = df.copy()

dashboard_df = dashboard_df.rename(
    columns={
        'age_years': 'Age',
        'daily_hours_clean': 'Daily Usage (Hours)',
        'monthly_data_gb_clean': 'Monthly Data Usage (GB)',
        'devices': 'Device',
        'purpose': 'Purpose'
    }
)

print("Dashboard dataset prepared!")

display(dashboard_df.head())

Dashboard dataset prepared!


,id,Device,Purpose,user_age,daily_hours,monthly_data_gb,Age,Daily Usage (Hours),Monthly Data Usage (GB)
0,ID001,Smartphone,Entertainment,4 years,8 hours,5 GB,NaN,8.0,5.0
1,ID002,Smartphone,Communication tool & learning media,18,7,50 GB,18.0,7.0,50.0
2,ID003,"Smartphone, Laptop, Tablet",Entertainment,18 years,6 hours,10 GB,18.0,6.0,10.0
3,ID004,Smartphone,Communication,18,9 hours,20 GB,18.0,9.0,20.0
4,ID005,Smartphone,Communication,18,6,Uses Wi-Fi,18.0,6.0,NaN


In [32]:
# 30. CREATE KPI DASHBOARD CARDS

from plotly.subplots import make_subplots

total_respondents = len(df)

average_age = df['age_years'].mean()

average_daily_usage = df['daily_hours_clean'].mean()

average_monthly_data = df['monthly_data_gb_clean'].mean()

kpi_fig = make_subplots(
    rows=1,
    cols=4,
    specs=[[{'type': 'indicator'}] * 4]
)

kpi_values = [
    ('Total Respondents', total_respondents, ''),
    ('Average Age', average_age, ' years'),
    ('Daily Usage', average_daily_usage, ' hours'),
    ('Monthly Data Usage', average_monthly_data, ' GB')
]

for index, (label, value, suffix) in enumerate(kpi_values, start=1):

    kpi_fig.add_trace(
        go.Indicator(
            mode='number',
            value=value,
            number={
                'suffix': suffix,
                'valueformat': '.2f'
            },
            title={'text': label}
        ),
        row=1,
        col=index
    )

kpi_fig.update_layout(
    title='Technology Usage Dashboard — Key Indicators',
    height=250,
    template='plotly_white'
)

kpi_fig.show()

In [33]:
# 31. CREATE DASHBOARD CHARTS

from plotly.subplots import make_subplots

dashboard_fig = make_subplots(
    rows=2,
    cols=2,
    subplot_titles=[
        'Frequently Used Devices',
        'Purpose of Technology Usage',
        'Age Distribution',
        'Daily Usage vs Monthly Data Usage'
    ],
    specs=[
        [{'type': 'bar'}, {'type': 'bar'}],
        [{'type': 'histogram'}, {'type': 'scatter'}]
    ],
    vertical_spacing=0.15,
    horizontal_spacing=0.12
)

# ------------------------------------------
# Chart 1: Device Distribution
# ------------------------------------------

device_counts = (
    df['devices']
    .value_counts()
    .reset_index()
)

device_counts.columns = ['Device', 'Count']

dashboard_fig.add_trace(
    go.Bar(
        x=device_counts['Device'],
        y=device_counts['Count'],
        name='Devices',
        marker=dict(
            color='#00BFFF',
            line=dict(color='black', width=1)
        ),
        text=device_counts['Count'],
        textposition='outside'
    ),
    row=1,
    col=1
)

# ------------------------------------------
# Chart 2: Purpose Distribution
# ------------------------------------------

purpose_counts = (
    df['purpose']
    .value_counts()
    .reset_index()
)

purpose_counts.columns = ['Purpose', 'Count']

dashboard_fig.add_trace(
    go.Bar(
        x=purpose_counts['Purpose'],
        y=purpose_counts['Count'],
        name='Purpose',
        marker=dict(
            color='#00CED1',
            line=dict(color='black', width=1)
        ),
        text=purpose_counts['Count'],
        textposition='outside'
    ),
    row=1,
    col=2
)

# ------------------------------------------
# Chart 3: Age Distribution
# ------------------------------------------

dashboard_fig.add_trace(
    go.Histogram(
        x=df['age_years'].dropna(),
        name='Age',
        nbinsx=15,
        marker=dict(
            color='#00BFFF',
            line=dict(color='black', width=1)
        )
    ),
    row=2,
    col=1
)

# ------------------------------------------
# Chart 4: Daily Usage vs Data Usage
# ------------------------------------------

scatter_data = df.dropna(
    subset=[
        'daily_hours_clean',
        'monthly_data_gb_clean'
    ]
)

dashboard_fig.add_trace(
    go.Scatter(
        x=scatter_data['daily_hours_clean'],
        y=scatter_data['monthly_data_gb_clean'],
        mode='markers',
        name='Respondents',
        marker=dict(
            size=10,
            color='#0077B6',
            line=dict(color='black', width=1)
        ),
        text=scatter_data['id'],
        hovertemplate=(
            'Respondent: %{text}<br>'
            'Daily Usage: %{x} hours<br>'
            'Monthly Data: %{y} GB'
            '<extra></extra>'
        )
    ),
    row=2,
    col=2
)

# ------------------------------------------
# Dashboard Layout
# ------------------------------------------

dashboard_fig.update_layout(
    title={
        'text': 'Technology Usage Behavior Dashboard',
        'x': 0.5,
        'xanchor': 'center'
    },
    template='plotly_white',
    height=850,
    showlegend=False
)

dashboard_fig.update_xaxes(
    title_text='Device',
    row=1,
    col=1
)

dashboard_fig.update_yaxes(
    title_text='Number of Respondents',
    row=1,
    col=1
)

dashboard_fig.update_xaxes(
    title_text='Purpose',
    row=1,
    col=2
)

dashboard_fig.update_yaxes(
    title_text='Number of Respondents',
    row=1,
    col=2
)

dashboard_fig.update_xaxes(
    title_text='Age (Years)',
    row=2,
    col=1
)

dashboard_fig.update_yaxes(
    title_text='Number of Respondents',
    row=2,
    col=1
)

dashboard_fig.update_xaxes(
    title_text='Daily Usage (Hours)',
    row=2,
    col=2
)

dashboard_fig.update_yaxes(
    title_text='Monthly Data Usage (GB)',
    row=2,
    col=2
)

dashboard_fig.show()

In [34]:
# 32. CORRELATION HEATMAP

correlation_columns = [
    'age_years',
    'daily_hours_clean',
    'monthly_data_gb_clean'
]

correlation_matrix = df[
    correlation_columns
].corr(method='pearson')

fig = px.imshow(
    correlation_matrix,
    text_auto='.2f',
    color_continuous_scale='RdBu_r',
    zmin=-1,
    zmax=1,
    title='Correlation Between Technology Usage Variables'
)

fig.update_layout(
    template='plotly_white',
    height=500
)

fig.show()

In [35]:
# 33. EXPORT CLEANED DATASET

output_filename = 'Technology_Usage_Cleaned.csv'

df.to_csv(
    output_filename,
    index=False
)

print("Cleaned dataset exported successfully!")
print("File name:", output_filename)

files.download(output_filename)

Cleaned dataset exported successfully!
File name: Technology_Usage_Cleaned.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [36]:
# 34. EXPORT INTERACTIVE DASHBOARD

dashboard_filename = 'Technology_Usage_Dashboard.html'

dashboard_fig.write_html(
    dashboard_filename,
    include_plotlyjs=True
)

print("Interactive dashboard exported successfully!")

files.download(dashboard_filename)

Interactive dashboard exported successfully!


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>